In [1]:
from pathlib import Path
from transformers import BertForTokenClassification, AutoModelForSequenceClassification, AutoTokenizer
import torch

In [2]:
ner_model = BertForTokenClassification.from_pretrained('ja_f1_best').eval()
tokenizer = AutoTokenizer.from_pretrained("daisaku-s/medtxt_ner_roberta")
re_model = AutoModelForSequenceClassification.from_pretrained('ja_clf80').eval()

idx2tag = ner_model.config.id2label
out_dir = Path('test')
TEST = Path('SMM4H_2024_Task_2_test')
id2label = ['O', 'CAUSED', 'TREATMENT_FOR']

In [3]:
for f in TEST.iterdir():
    if not f.stem.startswith('ja_'):
        with (out_dir / (f.stem + '.ann')).open('w') as output:
            output.write("")
        continue
    with f.open() as text:
        annotations = []
        data = text.readlines()[0]
        # ner
        with torch.inference_mode():
            vecs = tokenizer(data,
                             padding=True, 
                             truncation=True,
                             return_tensors="pt", max_length=512)
            ner_logits = ner_model(input_ids=vecs["input_ids"], attention_mask=vecs["attention_mask"])
            idx = torch.argmax(ner_logits.logits, dim=2).detach().cpu().numpy().tolist()[0]
            tokens = vecs.tokens()[1: -1]
        labels = [idx2tag[x] for x in idx][1:-1]
        prev_label = None
        prev_tag = ['', '']
        candidate = []
        start = 0
        for token, label in zip(tokens, labels):
            tag = label.split('-')
            if token.startswith('▁'):
                token = token[1:]
                start += 1
                if len(token) == 0:
                    continue
            if tag[0] == 'B':
                candidate= [(start, len(token) + start, token)]
            elif tag[0] != 'B' and len(prev_tag) > 1 and len(tag) > 1 and tag[1] == prev_tag[1]:
                candidate.append((start, len(token) + start, token))
            elif candidate:
                anno = ''.join(ctoken for s, e, ctoken in candidate)
                annotations.append((prev_tag[1], candidate[0][0] - 1, candidate[-1][1] - 1, anno))  # fix for start = -1
                candidate = []
            start += len(token)
            prev_tag = tag
        count = 1
        new_annotations = []
        for ann in annotations:
            new_annotations.append(f"T{count}\t{ann[0]} {ann[1]} {ann[2]}\t{ann[3]}\n")
            count += 1
        # find all spans between the named entities
        classcorpus = []
        for a in new_annotations:
            for b in new_annotations:
                aid, ameta, _ = a.split('\t')
                bid, bmeta, _ = b.split('\t')
                if aid == bid:
                    continue
                alabel, astart, aend = ameta.split(' ')
                blabel, bstart, bend = bmeta.split(' ')
                if (alabel == 'DRUG' and blabel !='DRUG') or (alabel == 'DISORDER' and blabel != 'DRUG'):
                    start = min(int(aend), int(bend))
                    end = max(int(astart), int(bstart))
                    classcorpus.append([aid, bid, data[start: end]])
        # relation extraction
        rcount = 1
        for a,b, span in classcorpus:
            if span.strip():
                encoded_input = tokenizer(span, return_tensors='pt', max_length=512)
                with torch.inference_mode():
                    output = re_model(**encoded_input).logits
                    class_id = output.argmax().item()
                    if class_id > 0:
                        new_annotations.append(f"R{rcount}\t{id2label[class_id]} Arg1:{a} Arg2:{b}\n")
                        rcount += 1
        # save output
        with (out_dir / (f.stem + '.ann')).open('w') as output:
            for ann in new_annotations:
                output.write(ann)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



||MicroP|MicroR|MicroF1|MacroP|MacroR|MacroF1|
|---|---:|---:|---:|---:|---:|---:|
|Task2a train dev|0.6293|0.3159|0.4206|0.5781|0.2911|0.3854
|Task2a train 80| 0.6408|0.3152|0.4226|0.5782|0.2888|0.3837
|Task2b train dev|0.0903|0.0191|0.0315|0.0299|0.0276|0.0218|
|Task2b train 80|0.0289|0.0150|0.0197|0.0097|0.0214|0.0104|

```
ja best and clf dev ner
,tp,fp,fn,precision,recall,f1
DISORDER,591,305,1159,0.6596,0.3377,0.4467
DRUG,301,116,663,0.7218,0.3122,0.4359
FUNCTION,84,154,292,0.3529,0.2234,0.2736
all,976,575,2114,0.6293,0.3159,0.4206

ja best and clf dev re
type,tp,fp,fn,precision,recall,f1,fpm,fnm
CAUSED|DISORDER|DISORDER,1,37,70,0.0263,0.0141,0.0183,30,56
CAUSED|DISORDER|FUNCTION,0,8,13,0.0000,0.0000,0.0000,7,12
CAUSED|DRUG|DISORDER,11,74,430,0.1294,0.0249,0.0418,36,399
CAUSED|DRUG|FUNCTION,2,22,9,0.0833,0.1818,0.1143,10,5
CAUSED|FUNCTION|DISORDER,0,0,15,0.0000,0.0000,0.0000,0,15
CAUSED|FUNCTION|FUNCTION,0,0,2,0.0000,0.0000,0.0000,0,2
TREATMENT_FOR|DRUG|DISORDER,0,0,172,0.0000,0.0000,0.0000,0,135
TREATMENT_FOR|DRUG|FUNCTION,0,0,9,0.0000,0.0000,0.0000,0,8
all,14,141,720,0.0903,0.0191,0.0315,83,632

ja f1 best clf 80 ner

,tp,fp,fn,precision,recall,f1
DISORDER,584,260,1166,0.6919,0.3337,0.4503
DRUG,311,124,653,0.7149,0.3226,0.4446
FUNCTION,79,162,297,0.3278,0.2101,0.2561
all,974,546,2116,0.6408,0.3152,0.4226

ja f1 best clf 80 re

CAUSED|DISORDER|DISORDER,1,93,70,0.0106,0.0141,0.0121,50,59
CAUSED|DISORDER|FUNCTION,0,39,13,0.0000,0.0000,0.0000,31,12
CAUSED|DRUG|DISORDER,8,166,433,0.0460,0.0181,0.0260,73,397
CAUSED|DRUG|FUNCTION,2,48,9,0.0400,0.1818,0.0656,28,5
CAUSED|FUNCTION|DISORDER,0,0,15,0.0000,0.0000,0.0000,0,15
CAUSED|FUNCTION|FUNCTION,0,0,2,0.0000,0.0000,0.0000,0,2
TREATMENT_FOR|DISORDER|DISORDER,0,6,0,0.0000,0.0000,0.0000,2,0
TREATMENT_FOR|DISORDER|FUNCTION,0,2,0,0.0000,0.0000,0.0000,1,0
TREATMENT_FOR|DRUG|DISORDER,0,15,172,0.0000,0.0000,0.0000,10,136
TREATMENT_FOR|DRUG|FUNCTION,0,0,9,0.0000,0.0000,0.0000,0,9
all,11,369,723,0.0289,0.0150,0.0197,195,635
```
